In [1]:
import argparse
import base64
import importlib.util
import json
import os
import sys

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from sklearn.metrics import average_precision_score, roc_auc_score

MIN_CLASS = 30
SEED = 42


In [2]:
# ------------------------- helpers -------------------------
def _looks_like_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read(2048).lstrip().startswith("{")
    except Exception:
        return False


def _load_from_notebook(path):
    import types
    with open(path, encoding="utf-8") as f:
        nb = json.load(f)
    cells = [("".join(c["source"]) if isinstance(c["source"], list) else c["source"])
             for c in nb.get("cells", []) if c.get("cell_type") == "code"]
    picked = [c for c in cells if "def build_xy(" in c]
    if not picked:
        raise FileNotFoundError(f"No cell defining build_xy() found in {path}; is this the 04 notebook?")
    body = []
    for src in picked[-1:]:         
        lines = [ln for ln in src.splitlines() if not ln.lstrip().startswith(("%", "!"))]
        tree = ast.parse("\n".join(lines))
        for node in tree.body:
            keep = isinstance(node, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.ClassDef))
            if isinstance(node, ast.Assign) and all(isinstance(t, ast.Name) for t in node.targets):
                try:
                    ast.literal_eval(node.value)
                    keep = True
                except Exception:
                    keep = False
            if keep:
                body.append(node)
    mod = types.ModuleType("models04")
    exec(compile(ast.Module(body=body, type_ignores=[]), path, "exec"), mod.__dict__)
    print(f"Loaded 04 code from notebook {os.path.basename(path)}")
    return mod


def load_04(path=None):
    """Find the 04 code: a .py file, a .ipynb, or (if 04 lives in this notebook) the notebook namespace."""
    cands = [path] if path else []
    if "__file__" in globals():
        cands.append(os.path.join(os.path.dirname(os.path.abspath(__file__)), "04_models.py"))
    cands.append(os.path.join(os.getcwd(), "04_models.py"))
    for c in cands:
        if c and os.path.exists(c):
            if str(c).endswith(".ipynb") or _looks_like_json(c):
                return _load_from_notebook(c)
            spec = importlib.util.spec_from_file_location("models04", c)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            return mod
    main_ns = sys.modules.get("__main__")
    need = ["prepare", "build_xy", "lag_values", "make_model"]
    if main_ns is not None and all(hasattr(main_ns, n) for n in need):
        print("Using the 04 functions already defined in this notebook")
        return main_ns
    raise FileNotFoundError(
        "04 code not found. Pass --models-script path/to/your_04_notebook.ipynb (or a .py file), "
        "or run the 04 cell in this same notebook first.")


def lift_roc(y, s):
    ok = ~np.isnan(s)
    y, s = y[ok], s[ok]
    npos = int(y.sum())
    if npos < MIN_CLASS or (len(y) - npos) < MIN_CLASS:
        return np.nan, np.nan, len(y), np.nan
    base = y.mean()
    return average_precision_score(y, s) / base, roc_auc_score(y, s), len(y), base


def attach(m4, lab, model_dir, H):
    path = os.path.join(model_dir, f"oof_H{16 * H}d.csv.gz")
    oof = pd.read_csv(path, dtype={"pid": str})
    drop = [c for c in ["y", "onset", "eval_all"] if c in oof.columns]
    d = lab.merge(oof.drop(columns=drop), on=["pid", "year", "k"], how="left")
    Z = lab.set_index(["pid", "t"])
    z_issue = m4.lag_values(Z, "z", d["pid"].to_numpy(), d["t"].to_numpy(), H + 1).astype(float)
    d["pers"] = -z_issue
    d["has_hist"] = ~np.isnan(z_issue)
    d["onset"] = d["has_hist"] & (z_issue >= -1.0)
    return d


def score_cols(d):
    return [c for c in d.columns if "|" in c] + ["pers"]


In [3]:
# ------------------------- 1. paired bootstrap by year -------------------------
def per_year_lifts(d, col, mask):
    out = {}
    for y, g in d[mask].groupby("year"):
        L, _, _, _ = lift_roc(g["y"].to_numpy(), g[col].to_numpy(float))
        if not np.isnan(L):
            out[y] = L
    return pd.Series(out)


def run_boot(d, H, B, rng, ref="pers"):
    rows = []
    if ref not in d.columns:
        return rows
    for subset, mask in [("all", d["has_hist"]), ("onset", d["onset"])]:
        pers = per_year_lifts(d, ref, mask)
        for col in [c for c in d.columns if "|" in c and c != ref]:
            m = per_year_lifts(d, col, mask)
            yrs = pers.index.intersection(m.index)
            if len(yrs) < 6:
                continue
            diff = (m[yrs] - pers[yrs]).to_numpy()
            boots = np.array([rng.choice(diff, len(diff), replace=True).mean() for _ in range(B)])
            try:
                p = wilcoxon(diff).pvalue
            except ValueError:
                p = np.nan
            rows.append(dict(horizon_days=16 * H, subset=subset, reference=ref, model=col, n_years=len(yrs),
                             mean_lift_model=m[yrs].mean(), mean_lift_reference=pers[yrs].mean(),
                             mean_diff=diff.mean(), ci_lo=np.percentile(boots, 2.5),
                             ci_hi=np.percentile(boots, 97.5), frac_years_better=(diff > 0).mean(),
                             wilcoxon_p=p))
    return rows



In [4]:
# ------------------------- 2. breakdowns -------------------------
def breakdown(d, H, by):
    """Skill by group, on rows where EVERY method has a score (so persistence is not judged on extra years)."""
    rows = []
    mcols = [c for c in d.columns if "|" in c]
    common = d[mcols].notna().all(axis=1) if mcols else True
    for subset, mask in [("all", d["has_hist"] & common), ("onset", d["onset"] & common)]:
        sub = d[mask]
        for gval, g in sub.groupby(by):
            for col in score_cols(d):
                L, R, n, base = lift_roc(g["y"].to_numpy(), g[col].to_numpy(float))
                rows.append(dict(horizon_days=16 * H, subset=subset, by=by, group=gval, method=col,
                                 n=n, base_rate=base, lift=L, roc_auc=R))
    return rows


def plot_breakdown(bd, H_days, out_png, model_cols):
    sub = bd[(bd["horizon_days"] == H_days) & (bd["subset"] == "all")]
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
    methods = [m for m in model_cols + ["pers"] if m in sub["method"].unique()]
    for i, by in enumerate(["k", "zone", "year"]):
        s = sub[sub["by"] == by]
        for m in methods:
            g = s[s["method"] == m].sort_values("group")
            ax[i].plot(g["group"], g["lift"], marker="o", ms=3, ls="--" if m == "pers" else "-",
                       label=("persistence" if m == "pers" else m))
        ax[i].axhline(1, color="k", lw=0.6)
        ax[i].set_xlabel({"k": "window (start DOY = 1 + 16k)", "zone": "latitude zone (0 south - 2 north)",
                          "year": "year"}[by])
        ax[i].set_ylabel("PR-AUC lift")
    ax[0].legend(fontsize=7)
    fig.suptitle(f"Skill breakdown at {H_days}-day lead (all rows)")
    fig.tight_layout()
    fig.savefig(out_png, dpi=150)
    plt.close(fig)


In [5]:
# ------------------------- 3. maps -------------------------
def cell_table(d, cells):
    return cells[["lon", "lat"]].copy()


def static_maps(d, cells, H, model_col, ref_col, out_png, ref_label="persistence"):
    """Per-cell skill maps: observed stress rate, ROC-AUC of the model, of the reference, and their difference."""
    mask = d["has_hist"] & d[model_col].notna()
    rows = []
    for cell, g in d[mask].groupby("cell"):
        y = g["y"].to_numpy()
        _, r_m, n, base = lift_roc(y, g[model_col].to_numpy(float))
        _, r_r, _, _ = lift_roc(y, g[ref_col].to_numpy(float))
        rows.append(dict(cell=cell, base=base, roc_model=r_m, roc_ref=r_r, n=n))
    c = pd.DataFrame(rows).set_index("cell").join(cells[["lon", "lat"]])
    c["d_model_ref"] = c["roc_model"] - c["roc_ref"]
    c.to_csv(out_png.replace(".png", ".csv"))

    fig, ax = plt.subplots(1, 4, figsize=(19, 4.8))
    panels = [("base", "Observed stress rate (NDVI-anomaly proxy)", "YlOrRd", None),
              ("roc_model", f"ROC-AUC, {model_col}, {16 * H} d lead", "viridis", (0.45, 0.8)),
              ("roc_ref", f"ROC-AUC, {ref_label}", "viridis", (0.45, 0.8)),
              ("d_model_ref", f"ROC-AUC: model minus {ref_label}", "RdBu", (-0.1, 0.1))]
    for a_, (col, title, cmap, lim) in zip(ax, panels):
        kw = dict(vmin=lim[0], vmax=lim[1]) if lim else {}
        sc = a_.scatter(c["lon"], c["lat"], c=c[col], s=18, cmap=cmap, marker="s", **kw)
        a_.set_title(title, fontsize=9)
        a_.set_xlabel("lon")
        plt.colorbar(sc, ax=a_, shrink=0.85)
    ax[0].set_ylabel("lat")
    fig.tight_layout()
    fig.savefig(out_png, dpi=150)
    plt.close(fig)
    return c


def uncertainty_summary(d, ca, cb, out_csv):
    """Does model disagreement flag where the predictions are less reliable? Skill by disagreement tercile."""
    m = d["has_hist"] & d[ca].notna() & d[cb].notna()
    g = d[m].copy()
    g["p"] = (g[ca] + g[cb]) / 2
    g["dis"] = (g[ca] - g[cb]).abs()
    g["tercile"] = pd.qcut(g["dis"], 3, labels=["low", "mid", "high"], duplicates="drop")
    rows = []
    for t_, x in g.groupby("tercile", observed=True):
        L, R, n, base = lift_roc(x["y"].to_numpy(), x["p"].to_numpy(float))
        rows.append(dict(disagreement=t_, n=n, mean_disagreement=x["dis"].mean(), base_rate=base,
                         lift=L, roc_auc=R))
    out = pd.DataFrame(rows)
    out.to_csv(out_csv, index=False)
    print("\nSkill of the averaged prediction by model-disagreement tercile "
          "(if disagreement is informative, skill should fall from low to high):")
    print(out.round(3).to_string(index=False))


In [7]:
HTML = r"""<!DOCTYPE html>
<html lang="en"><head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITLE__</title>
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.css">
<style>
:root{--ink:#1f2933;--muted:#667085;--line:#e4e7ec;--accent:#d9480f}
*{box-sizing:border-box}
html,body{height:100%;margin:0;font:14px/1.45 system-ui,-apple-system,"Segoe UI",Roboto,Arial,sans-serif;color:var(--ink)}
#map{position:absolute;inset:0}
.panel{position:absolute;z-index:1000;top:12px;left:12px;width:350px;max-height:calc(100% - 24px);overflow:auto;background:rgba(255,255,255,.95);backdrop-filter:blur(6px);border:1px solid var(--line);border-radius:14px;box-shadow:0 8px 28px rgba(16,24,40,.18);padding:14px 16px}
.panel h1{font-size:16px;line-height:1.25;margin:0 0 2px}
.sub{color:var(--muted);font-size:12px;margin-bottom:10px}
.seg{display:flex;border:1px solid var(--line);border-radius:10px;overflow:hidden;margin:6px 0 10px}
.seg button{flex:1;padding:8px 4px;border:0;background:#fff;cursor:pointer;font:inherit;font-size:12px;color:var(--ink)}
.seg button+button{border-left:1px solid var(--line)}
.seg button.on{background:var(--accent);color:#fff}
.seg button:disabled{opacity:.4;cursor:not-allowed}
.row{display:flex;align-items:center;gap:8px;margin:6px 0;flex-wrap:wrap}
.lbl{font-size:12px;color:var(--muted)}
button.pill{border:1px solid var(--line);background:#fff;border-radius:999px;padding:6px 12px;cursor:pointer;font:inherit;font-size:13px}
button.pill:hover{background:#f5f6f8}
select{font:inherit;font-size:12px;padding:4px 6px;border:1px solid var(--line);border-radius:8px;background:#fff}
input[type=range]{width:100%;accent-color:var(--accent);margin:2px 0}
#when{font-weight:600;font-size:13px;margin:2px 0 0}
.cards{display:grid;grid-template-columns:1fr 1fr;gap:8px;margin:10px 0}
.card{border:1px solid var(--line);border-radius:10px;padding:8px 10px;background:#fff}
.card b{display:block;font-size:18px;line-height:1.2}
.card span{font-size:11px;color:var(--muted)}
#legend .bar{height:10px;border-radius:5px;margin:4px 0 2px;border:1px solid var(--line)}
#legend .lab{display:flex;justify-content:space-between;font-size:11px;color:var(--muted)}
details{margin-top:10px;font-size:12px;color:#344054}
summary{cursor:pointer;font-weight:600}
details p{margin:6px 0}
.pp{min-width:262px}.pp .pt{font-weight:700;font-size:14px}.pp .ps{color:var(--muted);font-size:12px}
.pp .when{margin:8px 0 4px;font-weight:600;font-size:12px}
.bar2{display:grid;grid-template-columns:112px 1fr 44px;align-items:center;gap:6px;margin:3px 0;font-size:12px}
.bar2 b{text-align:right}
.track{height:8px;background:#eef0f3;border-radius:4px;overflow:hidden}.track i{display:block;height:100%}
.tag{display:inline-block;padding:1px 8px;border-radius:999px;font-size:11px;font-weight:600;color:#fff;margin-left:4px}
svg text{font:10px system-ui,sans-serif;fill:#667085}
.leaflet-popup-content{margin:12px 14px}
@media(max-width:640px){.panel{width:auto;right:12px;max-height:46%}}
</style></head><body>
<div id="map"></div>
<div class="panel">
  <h1>__TITLE__</h1>
  <div class="sub" id="sub"></div>
  <div class="seg" id="layers">
    <button data-layer="pred" class="on">Predicted risk</button>
    <button data-layer="obs">Observed (proxy)</button>
    <button data-layer="unc" id="uncBtn">Disagreement</button>
  </div>
  <div class="row">
    <button class="pill" id="play">&#9654; Play</button>
    <button class="pill" id="peak">Most stressed window</button>
    <select id="speed" title="Playback speed"><option value="800">Slow</option><option value="380" selected>Normal</option><option value="140">Fast</option></select>
  </div>
  <div id="when"></div>
  <input id="slider" type="range" min="0" max="0" value="0" step="1" aria-label="Time">
  <div class="row"><span class="lbl">Show cells at or above <b id="thv">0%</b></span></div>
  <input id="thr" type="range" min="0" max="90" value="0" step="5" aria-label="Threshold">
  <div class="cards">
    <div class="card"><b id="c1">-</b><span>cells with data</span></div>
    <div class="card"><b id="c2">-</b><span>mean predicted risk</span></div>
    <div class="card"><b id="c3">-</b><span>mean observed stress (proxy)</span></div>
    <div class="card"><b id="c4">-</b><span>cells predicted at 50% or more</span></div>
  </div>
  <div class="lbl" style="margin-bottom:2px">Area average over time <span style="color:#e8590c">&#9632; predicted</span> <span style="color:#222">&#9632; observed</span> (click to jump)</div>
  <div id="area" style="cursor:pointer"></div>
  <div id="legend"></div>
  <details>
    <summary>How to read this map</summary>
    <p><b>Click any cell</b> for its location, local skill and full time series. Arrow keys step through time; space plays or pauses.</p>
    <p>Each square is a 0.05&deg; rainfall cell (about 5 km). Values are the share of sample points in the cell that are stressed (observed) or predicted stressed.</p>
    <p><b>Observed</b> is a satellite proxy (an unusually low NDVI for the time of year), not ground truth. <b>Predicted</b> is __NOTE__ It is a stress risk, not a yield or damage forecast.</p>
    <p>Skill is real for same-window detection; useful lead time beyond about 16 days is not demonstrated. Use this to prioritise where to look, not as an alarm.</p>
  </details>
</div>
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.js"></script>
<script>
(function(){
"use strict";
const CELLS=__CELLS__, META=__META__, STEPS=__STEPS__, HASUNC=__HASUNC__;
const NC=CELLS.length, NS=STEPS.length;
function dec(b){const s=atob(b),a=new Uint8Array(s.length);for(let i=0;i<s.length;i++)a[i]=s.charCodeAt(i);return a;}
const DATA={pred:dec("__PRED__"),obs:dec("__OBS__"),unc:dec("__UNC__")};
const MON=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"];
const RAMPS={risk:["#ffffcc","#ffeda0","#fed976","#feb24c","#fd8d3c","#fc4e2a","#e31a1c","#b10026"],
             unc:["#f7fbff","#deebf7","#c6dbef","#9ecae1","#6baed6","#4292c6","#2171b5","#084594"]};
function rgb(h){return [1,3,5].map(i=>parseInt(h.substr(i,2),16));}
function mix(a,b,t){const x=rgb(a),y=rgb(b);return "rgb("+x.map((v,i)=>Math.round(v+(y[i]-v)*t)).join(",")+")";}
function ramp(t,r){const n=r.length-1,x=Math.max(0,Math.min(1,t))*n,i=Math.min(n-1,Math.floor(x));return mix(r[i],r[i+1],x-i);}
function val(l,i,j){const v=DATA[l][i*NC+j];return v===255?null:(l==="unc"?v/254*0.4:v/254);}
function fmtDate(d){return d.getUTCDate()+" "+MON[d.getUTCMonth()]+" "+d.getUTCFullYear();}
function span(si){const y=STEPS[si][0],k=STEPS[si][1];return fmtDate(new Date(Date.UTC(y,0,1+16*k)))+" to "+fmtDate(new Date(Date.UTC(y,0,16+16*k)));}
function pct(x){return x===null||x===undefined?"no data":Math.round(x*100)+"%";}
function esc(s){return String(s).replace(/[&<>"]/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));}

// ---- aggregates
const meanPred=[],meanObs=[];let gObs=0,gObsN=0;
for(let i=0;i<NS;i++){let sp=0,so=0,n=0;for(let j=0;j<NC;j++){const p=val("pred",i,j),o=val("obs",i,j);if(p!==null){sp+=p;n++;}if(o!==null){so+=o;gObs+=o;gObsN++;}}
  meanPred.push(n?sp/n:null);meanObs.push(n?so/n:null);}
const globalObs=gObsN?gObs/gObsN:0.2;
let peakStep=0;meanObs.forEach((v,i)=>{if(v!==null&&v>(meanObs[peakStep]||0))peakStep=i;});

// ---- state
let layer="pred",step=0,thr=0,sel=null,timer=null;
const m0=/t=(\d+)/.exec(location.hash||"");step=m0?Math.min(NS-1,+m0[1]):peakStep;

// ---- map
const map=L.map("map",{preferCanvas:true});
const carto=L.tileLayer("https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",{subdomains:"abcd",maxZoom:19,attribution:"&copy; OpenStreetMap contributors &copy; CARTO"});
const esri=L.tileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",{maxZoom:18,attribution:"Tiles &copy; Esri"});
carto.addTo(map);
L.control.layers({"Light map":carto,"Satellite":esri},null,{position:"bottomright"}).addTo(map);
const rend=L.canvas({padding:0.5});
function rad(){return Math.max(3.5,(map.getZoom()-6.2)*2.3);}
const LAYER_NAME={pred:"Predicted risk",obs:"Observed stress (proxy)",unc:"Model disagreement"};
const mk=CELLS.map((c,j)=>{
  const m=L.circleMarker([c[1],c[0]],{renderer:rend,radius:6,weight:0,fillOpacity:0,color:"#111"}).addTo(map);
  m.bindPopup(()=>popup(j),{maxWidth:340});
  m.bindTooltip(()=>esc(META[j][1]||"Cell")+": "+LAYER_NAME[layer]+" "+(layer==="unc"?(val("unc",step,j)===null?"no data":Math.round(val("unc",step,j)*100)+" pp"):pct(val(layer,step,j))),{sticky:true,direction:"top",opacity:0.95});
  m.on("click",()=>{sel=j;draw();});
  m.on("popupclose",()=>{if(sel===j){sel=null;draw();}});
  return m;});
map.fitBounds(L.latLngBounds(CELLS.map(c=>[c[1],c[0]])).pad(0.06));
map.on("zoomend",()=>{const r=rad();mk.forEach(m=>m.setRadius(r));});
mk.forEach(m=>m.setRadius(rad()));

// ---- charts
function chart(list,W,H,cur){
  const pl=26,pr=6,pt=6,pb=16,iw=W-pl-pr,ih=H-pt-pb;
  let mx=0.2;list.forEach(s=>s.v.forEach(x=>{if(x!==null&&x>mx)mx=x;}));mx=Math.min(1,Math.ceil(mx*1.05/0.2)*0.2);
  const X=i=>pl+(NS>1?i/(NS-1):0)*iw,Y=v=>pt+ih-(v/mx)*ih;
  let g='<svg viewBox="0 0 '+W+' '+H+'" width="100%" role="img">';
  [0,mx/2,mx].forEach(t=>{g+='<line x1="'+pl+'" x2="'+(W-pr)+'" y1="'+Y(t).toFixed(1)+'" y2="'+Y(t).toFixed(1)+'" stroke="#e4e7ec"/><text x="'+(pl-3)+'" y="'+(Y(t)+3).toFixed(1)+'" text-anchor="end">'+Math.round(t*100)+'</text>';});
  const years=[...new Set(STEPS.map(s=>s[0]))],every=Math.ceil(years.length/8);
  years.forEach((y,yi)=>{if(yi%every)return;const i=STEPS.findIndex(s=>s[0]===y);g+='<text x="'+X(i).toFixed(1)+'" y="'+(H-3)+'" text-anchor="middle">'+y+'</text>';});
  list.forEach(s=>{
    if(s.dots){s.v.forEach((x,i)=>{if(x!==null)g+='<circle cx="'+X(i).toFixed(1)+'" cy="'+Y(x).toFixed(1)+'" r="1.7" fill="'+s.color+'" opacity="0.75"/>';});}
    else{let d="",pen=false;s.v.forEach((x,i)=>{if(x===null){pen=false;return;}d+=(pen?"L":"M")+X(i).toFixed(1)+" "+Y(x).toFixed(1);pen=true;});
      g+='<path d="'+d+'" fill="none" stroke="'+s.color+'" stroke-width="1.8" stroke-linejoin="round"/>';}
  });
  g+='<line x1="'+X(cur).toFixed(1)+'" x2="'+X(cur).toFixed(1)+'" y1="'+pt+'" y2="'+(pt+ih)+'" stroke="#111" stroke-dasharray="3 3"/>';
  return g+'</svg>';
}
function drawArea(){
  document.getElementById("area").innerHTML=chart([{v:meanPred,color:"#e8590c"},{v:meanObs,color:"#222"}],318,104,step);
}
document.getElementById("area").addEventListener("click",ev=>{
  const r=ev.currentTarget.getBoundingClientRect(),W=318,pl=26,pr=6;
  const x=(ev.clientX-r.left)/r.width*W;
  setStep(Math.round((x-pl)/(W-pl-pr)*(NS-1)));
});

// ---- popup
function popup(j){
  const m=META[j],p=val("pred",step,j),o=val("obs",step,j),u=HASUNC?val("unc",step,j):null;
  const ps=[],os=[];let sp=0,so=0,n=0;
  for(let i=0;i<NS;i++){const a=val("pred",i,j),b=val("obs",i,j);ps.push(a);os.push(b);if(a!==null){sp+=a;}if(b!==null){so+=b;n++;}}
  const mp=n?sp/n:null,mo=n?so/n:null;
  function bar(label,v,color,scale){return '<div class="bar2"><span>'+label+'</span><div class="track"><i style="width:'+(v===null?0:Math.min(100,v*100*(scale||1)))+'%;background:'+color+'"></i></div><b>'+(v===null?"-":Math.round(v*100)+"%")+'</b></div>';}
  let tag="";if(p!==null){const x=p/globalObs;tag='<span class="tag" style="background:'+(x>=1.5?"#c92a2a":x>=1?"#e8590c":"#2b8a3e")+'">'+(x>=1.5?"High":x>=1?"Above average":"Below average")+'</span>';}
  let h='<div class="pp"><div class="pt">'+esc(m[1]||"Cell")+tag+'</div>'
   +'<div class="ps">Zone '+m[2]+' (0 south, 2 north) &middot; '+m[0]+' sample point'+(m[0]===1?'':'s')+' &middot; '+CELLS[j][1].toFixed(2)+'&deg;N, '+CELLS[j][0].toFixed(2)+'&deg;E</div>'
   +'<div class="when">'+STEPS[step][0]+' &middot; '+span(step)+'</div>'
   +bar("Predicted risk",p,"#e8590c")+bar("Observed (proxy)",o,"#222");
  if(HASUNC)h+=bar("Model disagreement",u,"#2171b5",2.5);
  if(p!==null)h+='<div class="ps" style="margin-top:2px">Predicted risk is '+(p/globalObs).toFixed(1)+'&times; the average stress rate across the map ('+Math.round(globalObs*100)+'%).</div>';
  h+='<div class="when">All windows in this cell</div>'+chart([{v:ps,color:"#e8590c"},{v:os,color:"#222",dots:true}],300,96,step)
   +'<div class="ps"><span style="color:#e8590c">&#9632;</span> predicted &nbsp; <span style="color:#222">&#9679;</span> observed &nbsp; mean predicted '+pct(mp)+', observed '+pct(mo)+'</div>';
  if(m[3]!==null&&m[3]!==undefined){h+='<div class="ps" style="margin-top:4px">Local skill (ROC-AUC, 1 = perfect, 0.5 = chance): model <b>'+m[3].toFixed(2)+'</b>'+(m[4]!==null&&m[4]!==undefined?' vs persistence '+m[4].toFixed(2):'')+'</div>';}
  return h+'</div>';
}

// ---- draw
const L2={pred:"Predicted stress risk",obs:"Observed stress rate (NDVI-anomaly proxy)",unc:"Disagreement between two model types"};
function legend(){
  const r=layer==="unc"?RAMPS.unc:RAMPS.risk;
  const lab=layer==="unc"?["0","20","40 percentage points"]:["0%","50%","100%"];
  document.getElementById("legend").innerHTML='<div class="lbl">'+L2[layer]+'</div><div class="bar" style="background:linear-gradient(90deg,'+r.join(",")+')"></div><div class="lab"><span>'+lab[0]+'</span><span>'+lab[1]+'</span><span>'+lab[2]+'</span></div>';
}
function draw(){
  const lo=thr/100,r=layer==="unc"?RAMPS.unc:RAMPS.risk;
  let n=0,sp=0,so=0,no=0,hi=0;
  for(let j=0;j<NC;j++){
    const v=val(layer,step,j),p=val("pred",step,j),o=val("obs",step,j);
    if(p!==null){n++;sp+=p;if(p>=0.5)hi++;}if(o!==null){so+=o;no++;}
    if(v===null){mk[j].setStyle({fillOpacity:0,weight:0});continue;}
    const show=layer==="unc"||v>=lo;
    mk[j].setStyle({fillColor:ramp(layer==="unc"?v/0.4:v,r),fillOpacity:show?0.9:0.05,weight:j===sel?2.5:0,opacity:1});
    if(j===sel)mk[j].bringToFront();
  }
  document.getElementById("c1").textContent=n;
  document.getElementById("c2").textContent=n?Math.round(sp/n*100)+"%":"-";
  document.getElementById("c3").textContent=no?Math.round(so/no*100)+"%":"-";
  document.getElementById("c4").textContent=n?Math.round(hi/n*100)+"%":"-";
  document.getElementById("when").textContent=STEPS[step][0]+"  \u00b7  "+span(step);
  document.getElementById("slider").value=step;
  document.getElementById("thv").textContent=thr+"%";
  drawArea();legend();
  if(sel!==null&&mk[sel].isPopupOpen())mk[sel].setPopupContent(popup(sel));
  try{history.replaceState(null,"","#t="+step);}catch(e){}
}
function setStep(i){step=Math.max(0,Math.min(NS-1,i));draw();}
function stop(){if(timer){clearInterval(timer);timer=null;document.getElementById("play").innerHTML="&#9654; Play";}}
function play(){if(timer){stop();return;}document.getElementById("play").innerHTML="&#10074;&#10074; Pause";
  timer=setInterval(()=>setStep((step+1)%NS),+document.getElementById("speed").value);}

// ---- controls
document.querySelectorAll("#layers button").forEach(b=>b.addEventListener("click",()=>{
  layer=b.dataset.layer;document.querySelectorAll("#layers button").forEach(x=>x.classList.toggle("on",x===b));draw();}));
if(!HASUNC){const u=document.getElementById("uncBtn");u.disabled=true;u.title="Run 04 with two model types to enable";}
document.getElementById("slider").max=NS-1;
document.getElementById("slider").addEventListener("input",e=>{stop();setStep(+e.target.value);});
document.getElementById("thr").addEventListener("input",e=>{thr=+e.target.value;draw();});
document.getElementById("play").addEventListener("click",play);
document.getElementById("peak").addEventListener("click",()=>{stop();setStep(peakStep);});
document.getElementById("speed").addEventListener("change",()=>{if(timer){stop();play();}});
document.addEventListener("keydown",e=>{
  if(e.target.tagName==="SELECT")return;
  if(e.key==="ArrowRight"){stop();setStep(step+1);}else if(e.key==="ArrowLeft"){stop();setStep(step-1);}
  else if(e.key===" "&&e.target.tagName!=="BUTTON"){e.preventDefault();play();}});
document.getElementById("sub").textContent=STEPS[0][0]+" to "+STEPS[NS-1][0]+" \u00b7 "+NS+" windows \u00b7 "+NC+" cells \u00b7 out-of-fold predictions";
draw();
})();
</script></body></html>
"""


def build_html(d, cells, H, score_col, out_html, title, unc_cols=None, cell_skill=None):
    sub = d[d["has_hist"] & d[score_col].notna()]
    agg = sub.groupby(["cell", "year", "k"]).agg(pred=(score_col, "mean"), obs=("y", "mean")).reset_index()
    steps = agg[["year", "k"]].drop_duplicates().sort_values(["year", "k"]).reset_index(drop=True)
    cell_ids = sorted(agg["cell"].unique())
    cidx = {c: i for i, c in enumerate(cell_ids)}
    sidx = {(int(r.year), int(r.k)): i for i, r in steps.iterrows()}
    NC, NS = len(cell_ids), len(steps)
    P = np.full((NS, NC), 255, dtype=np.uint8)
    O = np.full((NS, NC), 255, dtype=np.uint8)
    ci = agg["cell"].map(cidx).to_numpy()
    si = np.array([sidx[(int(y), int(k))] for y, k in zip(agg["year"], agg["k"])])
    P[si, ci] = np.clip(np.round(agg["pred"].to_numpy() * 254), 0, 254).astype(np.uint8)
    O[si, ci] = np.clip(np.round(agg["obs"].to_numpy() * 254), 0, 254).astype(np.uint8)
    U = np.full((NS, NC), 255, dtype=np.uint8)
    if unc_cols:
        ca, cb = unc_cols
        su = d[d["has_hist"] & d[ca].notna() & d[cb].notna()].copy()
        su["dis"] = (su[ca] - su[cb]).abs()
        au = su.groupby(["cell", "year", "k"])["dis"].mean().reset_index()
        au = au[au["cell"].isin(cidx.keys())]
        ui = au["cell"].map(cidx).to_numpy()
        us = np.array([sidx.get((int(y), int(k)), -1) for y, k in zip(au["year"], au["k"])])
        okk = us >= 0
        U[us[okk], ui[okk]] = np.clip(np.round(au["dis"].to_numpy()[okk] / 0.4 * 254), 0, 254).astype(np.uint8)
    xy = cells.loc[cell_ids, ["lon", "lat"]].round(3).values.tolist()
    # per-cell metadata for the click popup: [n points, LGA, zone, model ROC-AUC, reference ROC-AUC]
    g = d[d["cell"].isin(cell_ids)].groupby("cell")
    npts = g["pid"].nunique()
    lga = g["lga"].agg(lambda x: x.dropna().mode().iloc[0] if x.notna().any() else "") if "lga" in d.columns else None
    zone = g["zone"].agg(lambda x: int(x.mode().iloc[0])) if "zone" in d.columns else None

    def _f(v):
        return None if v is None or (isinstance(v, float) and np.isnan(v)) else round(float(v), 3)

    meta = []
    for c_ in cell_ids:
        rm = rr = None
        if cell_skill is not None and c_ in cell_skill.index:
            rm, rr = _f(cell_skill.loc[c_, "roc_model"]), _f(cell_skill.loc[c_, "roc_ref"])
        meta.append([int(npts.get(c_, 0)), str(lga.get(c_, "")) if lga is not None else "",
                     int(zone.get(c_, 0)) if zone is not None else 0, rm, rr])
    note = (f"the out-of-fold probability from model <i>{score_col}</i> at a {16 * H}-day lead: each prediction "
            "comes from a model that never saw that year group or that spatial block.")
    html = (HTML.replace("__TITLE__", title)
            .replace("__CELLS__", json.dumps(xy))
            .replace("__STEPS__", json.dumps([[int(a), int(b)] for a, b in steps[["year", "k"]].values]))
            .replace("__PRED__", base64.b64encode(P.tobytes()).decode())
            .replace("__OBS__", base64.b64encode(O.tobytes()).decode())
            .replace("__UNC__", base64.b64encode(U.tobytes()).decode())
            .replace("__META__", json.dumps(meta))
            .replace("__HASUNC__", "true" if unc_cols else "false")
            .replace("__NOTE__", note))
    with open(out_html, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"Saved {out_html}  ({os.path.getsize(out_html) / 1e6:.1f} MB, {NC} cells x {NS} steps)")

In [6]:
# ------------------------- 4. SHAP -------------------------
def run_shap(m4, feat, lab, H, sets, model_name, out_dir, n_fit=150_000, n_explain=5_000, explain_min_year=None):
    import shap
    rng = np.random.default_rng(SEED)
    Xc, Xn = m4.build_xy(feat, lab, H)
    X_sets = {"climate": Xc, "ndvi": pd.concat([Xn, Xc[["k"]]], axis=1),
              "climate+ndvi": pd.concat([Xc, Xn], axis=1)}
    if any("s1" in sn for sn in sets):
        Xs = m4.build_s1(feat, lab, H)
        kcol = Xc[["k"]]
        X_sets.update({"s1": pd.concat([Xs, kcol], axis=1),
                       "ndvi+s1": pd.concat([Xn, Xs, kcol], axis=1),
                       "climate+s1": pd.concat([Xc, Xs], axis=1),
                       "climate+ndvi+s1": pd.concat([Xc, Xn, Xs], axis=1)})
    y = lab["y"].to_numpy()
    ex_pool = np.flatnonzero((lab["year"] >= explain_min_year).to_numpy()) if explain_min_year else np.arange(len(lab))
    try:
        import xgboost  # noqa: F401
        mname = "xgb" if model_name == "xgb" else model_name
    except ImportError:
        mname = "rf"
        print("xgboost not installed -> SHAP with RandomForest")
    if mname == "hgb":
        mname = "rf"
    all_rows = []
    for sname in sets:
        X = X_sets[sname]
        idx = rng.choice(len(X), min(n_fit, len(X)), replace=False)
        ytr = y[idx]
        model, needs_fill = m4.make_model(mname, (len(ytr) - ytr.sum()) / max(ytr.sum(), 1))
        Xtr = X.iloc[idx]
        Xtr = Xtr.fillna(-999) if needs_fill else Xtr
        model.fit(Xtr, ytr)
        ex_idx = rng.choice(ex_pool, min(n_explain, len(ex_pool)), replace=False)
        Xe = X.iloc[ex_idx]
        Xe = Xe.fillna(-999) if needs_fill else Xe
        sv = shap.TreeExplainer(model).shap_values(Xe)
        if isinstance(sv, list):
            sv = sv[1]
        sv = np.asarray(sv)
        if sv.ndim == 3:
            sv = sv[:, :, 1]
        imp = pd.DataFrame({"feature": X.columns, "mean_abs_shap": np.abs(sv).mean(axis=0)})
        imp["variable"] = imp["feature"].str.replace(r"_L\d+$", "", regex=True)
        imp["lag_windows"] = imp["feature"].str.extract(r"_L(\d+)$")[0].astype(float)
        imp["horizon_days"] = 16 * H
        imp["set"] = sname
        all_rows.append(imp)

        by_var = imp.groupby("variable")["mean_abs_shap"].sum().sort_values()
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
        by_var.plot.barh(ax=ax[0])
        ax[0].set_title(f"Total mean |SHAP| by variable ({sname}, {16 * H} d lead)", fontsize=10)
        top = by_var.sort_values(ascending=False).index[:5]
        for v in top:
            g = imp[imp["variable"] == v].sort_values("lag_windows")
            if g["lag_windows"].notna().any():
                ax[1].plot(g["lag_windows"], g["mean_abs_shap"], marker="o", label=v)
        ax[1].set_xlabel("feature lag (windows before the target window)")
        ax[1].set_ylabel("mean |SHAP|")
        ax[1].legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, f"shap_{sname.replace('+', '_')}_H{16 * H}d.png"), dpi=150)
        plt.close(fig)
    pd.concat(all_rows).to_csv(os.path.join(out_dir, f"shap_importance_H{16 * H}d.csv"), index=False)

In [9]:

# ------------------------- main -------------------------
def run(a):
    m4 = load_04(a.models_script)
    os.makedirs(a.out_dir, exist_ok=True)
    rng = np.random.default_rng(SEED)
    feat, lab, cells = m4.prepare(a.feat_dir, a.label_dir, a.feat_file, a.min_year, a.target)

    boot_rows, bd_rows = [], []
    d_cache = {}
    for H in a.horizons:
        try:
            d = attach(m4, lab, a.model_dir, H)
        except FileNotFoundError:
            print(f"No OOF file for H={H}; skipping")
            continue
        d_cache[H] = d
        if "boot" in a.steps:
            for ref in a.boot_refs:
                r = "pers" if ref == "pers" else ref
                boot_rows += run_boot(d, H, a.boot_n, rng, r)
        if "breakdown" in a.steps:
            for by in ["k", "zone", "year"]:
                bd_rows += breakdown(d, H, by)

    if boot_rows:
        bt = pd.DataFrame(boot_rows)
        bt.to_csv(os.path.join(a.out_dir, "bootstrap_vs_persistence.csv"), index=False)
        print("\nPaired bootstrap by year (mean lift difference of model minus reference, 95% CI):")
        print(bt[["horizon_days", "subset", "reference", "model", "n_years", "mean_diff", "ci_lo", "ci_hi",
                  "frac_years_better", "wilcoxon_p"]].round(3).to_string(index=False))

    if bd_rows:
        bd = pd.DataFrame(bd_rows)
        bd.to_csv(os.path.join(a.out_dir, "breakdowns.csv"), index=False)
        cols = sorted({c for dd in d_cache.values() for c in dd.columns if "|" in c})
        for H in d_cache:
            plot_breakdown(bd, 16 * H, os.path.join(a.out_dir, f"breakdown_H{16 * H}d.png"), cols)

    if "maps" in a.steps:
        H = a.map_horizon
        if H not in d_cache:
            d_cache[H] = attach(m4, lab, a.model_dir, H)
        d = d_cache[H]
        avail = [c.split("|")[0] for c in d.columns if c.endswith("|" + a.model)]
        pref = [a.map_set] if a.map_set else ["climate+ndvi+s1", "climate+ndvi", "ndvi+s1", "climate", "ndvi"]
        map_set = next((p_ for p_ in pref if p_ in avail), None)
        if map_set is None:
            raise ValueError(f"No OOF column for the map set. Available sets for model '{a.model}': {avail}")
        col = f"{map_set}|{a.model}"
        ref_col = "pers" if a.map_ref == "pers" else f"{a.map_ref}|{a.model}"
        ref_label = "persistence" if a.map_ref == "pers" else a.map_ref
        print(f"Maps: model = {col}, reference = {ref_label}, lead = {16 * H} days")
        cell_skill = static_maps(d, cells, H, col, ref_col, os.path.join(a.out_dir, f"skill_maps_H{16 * H}d.png"), ref_label)
        unc_cols = None
        if len(a.unc_models) >= 2:
            ca, cb = f"{map_set}|{a.unc_models[0]}", f"{map_set}|{a.unc_models[1]}"
            if ca in d.columns and cb in d.columns:
                unc_cols = (ca, cb)
                uncertainty_summary(d, ca, cb, os.path.join(a.out_dir, f"uncertainty_H{16 * H}d.csv"))
            else:
                print(f"No uncertainty layer: need OOF columns {ca} and {cb} (run 04 with --models "
                      f"{a.unc_models[0]} {a.unc_models[1]}).")
        build_html(d, cells, H, col, os.path.join(a.out_dir, "hotspot_map.html"),
                   f"Crop-stress detection and early warning, SW Nigeria: {16 * H}-day lead ({map_set})",
                   unc_cols=unc_cols, cell_skill=cell_skill)

    if "shap" in a.steps:
        try:
            import shap  # noqa: F401
        except ImportError:
            print("shap not installed (pip install shap); skipping SHAP step")
        else:
            first = next(iter(d_cache.values())) if d_cache else attach(m4, lab, a.model_dir, a.shap_horizons[0])
            avail = [c.split("|")[0] for c in first.columns if c.endswith("|" + a.model)]
            sets = a.shap_sets or [x for x in ["climate+ndvi", "climate+ndvi+s1"] if x in avail] or avail[:2]
            print(f"SHAP sets: {sets}")
            for H in a.shap_horizons:
                run_shap(m4, feat, lab, H, sets, a.model, a.out_dir, explain_min_year=a.eval_min_year)
                print(f"SHAP done for H={H}")


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--feat-dir", required=True)
    ap.add_argument("--label-dir", required=True)
    ap.add_argument("--feat-file", default="features_long",
                    help="feature table name; use features_long_s1 for models trained with Sentinel-1 sets")
    ap.add_argument("--min-year", type=int, default=None,
                    help="must match the --min-year used in 04 (2016 for the Sentinel-1 test)")
    ap.add_argument("--model-dir", required=True, help="folder with oof_H*d.csv.gz from 04_models.py")
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--models-script", default=None, help="path to 04_models.py if not alongside")
    ap.add_argument("--model", default="xgb", help="model name used in the OOF columns (xgb, rf, hgb)")
    ap.add_argument("--horizons", nargs="*", type=int, default=[0, 1, 2, 3])
    ap.add_argument("--steps", nargs="*", default=["boot", "breakdown", "maps", "shap"],
                    choices=["boot", "breakdown", "maps", "shap"])
    ap.add_argument("--boot-n", type=int, default=2000)
    ap.add_argument("--boot-refs", nargs="*", default=["pers", "ndvi|xgb"],
                    help="reference columns for the paired bootstrap; 'pers' = persistence, 'ndvi|xgb' = NDVI-only model")
    ap.add_argument("--map-horizon", type=int, default=1, help="in 16-day windows")
    ap.add_argument("--map-set", default=None,
                    help="model set to map; default = best available (climate+ndvi+s1, climate+ndvi, ...)")
    ap.add_argument("--unc-models", nargs="*", default=["xgb", "rf"],
                    help="two model types whose disagreement is mapped as the uncertainty layer")
    ap.add_argument("--target", default="stress", choices=["stress", "sustained"],
                    help="must match the --target used in 04")
    ap.add_argument("--map-ref", default="pers",
                    help="reference for the skill-difference map: 'pers' (persistence) or a set name such as 'ndvi'")
    ap.add_argument("--eval-min-year", type=int, default=None,
                    help="use the same value as in 04 (2016 for Sentinel-1); SHAP then explains only those years")
    ap.add_argument("--shap-horizons", nargs="*", type=int, default=[0, 1])
    ap.add_argument("--shap-sets", nargs="*", default=None,
                    help="default: climate+ndvi and climate+ndvi+s1 if present in the OOF files")
    a = ap.parse_args(argv)
    return run(a)


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

In [6]:
main(["--feat-dir", r"D:\\crop_stress\\features", "--label-dir", r"D:\\crop_stress\\label",
        "--model-dir", r"D:\\crop_stress\\models", "--out-dir", r"D:\\crop_stress\\analysis"])

Loaded 04 code from notebook models_2.ipynb


C:\Users\DELL\anaconda3\envs\geospatial\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(


Rows: 591,104  points: 3,185  cells: 572  base stress rate: 0.208
Maps: model = climate+ndvi+s1|xgb, reference = persistence, lead = 0 days
Saved crop_stress\analysis_s1b\hotspot_map.html  (0.2 MB, 572 cells x 90 steps)
SHAP sets: ['climate+ndvi', 'climate+ndvi+s1']
SHAP done for H=0
SHAP done for H=1


In [2]:
main(["--feat-dir", r"crop_stress\features", "--feat-file", "features_long_s1",
      "--label-dir", r"crop_stress\label", "--model-dir", r"crop_stress\models_sust", "--out-dir", 
      r"crop_stress\analysis_unc",
      "--model", "xgb", "--unc-models", "xgb", "rf", "--map-horizon", "0", "--eval-min-year", "2016",
      "--steps", "maps", "--models-script", r"models_2.ipynb"])

Loaded 04 code from notebook models_2.ipynb


C:\Users\DELL\anaconda3\envs\geospatial\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(


Rows: 591,104  points: 3,185  cells: 572  base stress rate: 0.208
No OOF file for H=3; skipping
Maps: model = climate+ndvi+s1|xgb, reference = persistence, lead = 0 days
No uncertainty layer: need OOF columns climate+ndvi+s1|xgb and climate+ndvi+s1|rf (run 04 with --models xgb rf).
Saved crop_stress\analysis_unc\hotspot_map.html  (0.2 MB, 572 cells x 80 steps)
